# ARC3 HDS HOLD baseline

現行HDS候補を照会し、確定作用が存在しない場合は操作せず終了する基線です。Save & Run Allはdummy parquetを作るだけで、Competition Rerunの結果ではありません。HDS/接続ソースのみを埋め込み、入力・事前記憶・認証情報は含めません。

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
import base64, hashlib, io, json, sys, zipfile
from pathlib import Path, PurePosixPath

_BUNDLE_SHA256 = "bd3c5e2ff0a3e4365e9bf9b5b42bdbd081bca49ceb69bf56f038f7f2e6e9c84e"
_BUNDLE_B64 = """..."""
payload = base64.b64decode(_BUNDLE_B64, validate=True)
if hashlib.sha256(payload).hexdigest() != _BUNDLE_SHA256:
    raise RuntimeError("ARC3 source bundle SHA256 mismatch")
root = Path("/kaggle/working/arc3-hds-source")
root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("ARC3 source bundle ZIP is corrupt")
    names = archive.namelist()
    for name in names:
        path = Path(name)
        if path.is_absolute() or ".." in path.parts or not (
            name.startswith("接続/") or name.startswith("HDS/学習系統/v0.4.2/hds学習系統/")
            or name == "ARC3-SOURCE-MANIFEST.json"
        ):
            raise RuntimeError(f"Unexpected source bundle path: {name}")
    archive.extractall(root)
manifest = json.loads((root / "ARC3-SOURCE-MANIFEST.json").read_text(encoding="utf-8"))
for item in manifest["含有"]:
    path = root / PurePosixPath(item["パス"])
    raw = path.read_bytes()
    if len(raw) != item["bytes"] or hashlib.sha256(raw).hexdigest() != item["sha256"]:
        raise RuntimeError(f"Source file verification failed: {item['パス']}")
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "HDS/学習系統/v0.4.2"))
import compileall
if not compileall.compile_dir(str(root / "接続"), quiet=1, force=True):
    raise RuntimeError("Connection source compile failed")
if not compileall.compile_dir(str(root / "HDS/学習系統/v0.4.2/hds学習系統"), quiet=1, force=True):
    raise RuntimeError("HDS source compile failed")
from hds学習系統 import HDS学習系統
from 接続.ARC3境界 import 観測する
from 接続.共通観測経路 import 公開観測から候補を照会する
print("Verified source files:", len(manifest["含有"]))
print("Verified HDS:", HDS学習系統.__module__)
    

In [ ]:
%%writefile /tmp/my_agent.py
"""現行のHDS候補を確定扱いせず停止する、初回提出用の棄権基線。

最初の公開観測がNOT_PLAYEDなら、RESETを一度だけ送り次のFrameを取得する。
SDKが最初からNOT_FINISHEDを返す場合は何も送らず、観測をHDS候補経路へ渡して終了する。
現行repoに作用認可発行元がないため、追加のGameActionは返さない。
"""
from __future__ import annotations

import logging
from typing import Any

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState
from hds学習系統 import HDS学習系統

from 接続.ARC3境界 import 境界違反, 観測する
from 接続.共通観測経路 import 公開観測から候補を照会する

_LOG = logging.getLogger(__name__)


class MyAgent(Agent):
    """公開観測を候補照会し、未認可候補では直ちに終了する。"""

    MAX_ACTIONS = 1

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self._hds = HDS学習系統()
        self._last_hold: dict[str, Any] | None = None

    @property
    def last_hold(self) -> dict[str, Any] | None:
        """試験・診断向けの要約。盤面、秘密、候補内容は保持しない。"""
        return None if self._last_hold is None else dict(self._last_hold)

    def _現在のSDK観測(self, fallback: FrameData) -> FrameData | None:
        if self.arc_env is None:
            return fallback
        raw = getattr(self.arc_env, "observation_space", None)
        if raw is None:
            return None
        return self._convert_raw_frame_data(raw)

    def _HDS照会後に停止する(self, frames: list[FrameData], frame: FrameData) -> None:
        try:
            observation = 観測する(frame, 観測通番=max(0, len(frames) - 1))
            candidate = 公開観測から候補を照会する(self._hds, observation)
            self._last_hold = {
                "状態": "HOLD",
                "観測指紋": observation.指紋,
                "候補経路状態": candidate.get("状態"),
                "次作用候補数": len(candidate.get("次作用候補群", [])),
                "候補側申告実行許可": candidate.get("実行許可") is True,
            }
            _LOG.info(
                "ARC3 baseline HOLD: candidate_state=%s candidate_count=%d permission_claim=%s",
                self._last_hold["候補経路状態"],
                self._last_hold["次作用候補数"],
                self._last_hold["候補側申告実行許可"],
            )
        except Exception as exc:
            self._last_hold = {
                "状態": "HOLD",
                "理由": type(exc).__name__,
                "候補側申告実行許可": False,
            }
            _LOG.warning("ARC3 baseline HOLD: observation query failed (%s)", type(exc).__name__)

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        if latest_frame.state is GameState.WIN:
            return True

        # Agent.main checks its placeholder frame before passing the current SDK observation
        # to choose_action. Inspect the SDK observation here to avoid forcing a RESET or move.
        if latest_frame.state is GameState.NOT_PLAYED and self.action_counter == 0:
            current = self._現在のSDK観測(latest_frame)
            if current is None:
                self._last_hold = {
                    "状態": "HOLD",
                    "理由": "初回SDK観測欠落",
                    "候補側申告実行許可": False,
                }
                return True
            if current.state is GameState.NOT_PLAYED:
                return False
            if current.state is GameState.NOT_FINISHED:
                self._HDS照会後に停止する(frames, current)
            return True

        if latest_frame.state is GameState.NOT_FINISHED:
            self._HDS照会後に停止する(frames, latest_frame)
        return True

    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        if latest_frame.state is GameState.NOT_PLAYED and self.action_counter == 0:
            return GameAction.RESET
        raise 境界違反("現行HDS確定作用がないため、初期化以外のActionは返せません。")



In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    subprocess.run(
        ["curl", "--fail", "--retry", "999", "--retry-all-errors",
         "--retry-delay", "5", "--retry-max-time", "600",
         "http://gateway:8001/api/games"],
        check=True,
    )
    framework = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    working = Path("/kaggle/working/ARC-AGI-3-Agents")
    shutil.copytree(framework, working, dirs_exist_ok=True)
    shutil.copyfile(
        "/tmp/my_agent.py",
        working / "agents/templates/my_agent.py",
    )
    (working / "agents/__init__.py").write_text(
        """from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""",
        encoding="utf-8",
    )
    (working / ".env").write_text(
        """SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""",
        encoding="utf-8",
    )
    env = os.environ.copy()
    env["PYTHONPATH"] = "/kaggle/working/arc3-hds-source:/kaggle/working/arc3-hds-source/HDS/学習系統/v0.4.2"
    env["MPLBACKEND"] = "agg"
    subprocess.run(["python", "main.py", "--agent", "myagent"], cwd=working, env=env, check=True)
else:
    # Save & Run Allのdummy parquet。Competition Rerunの得点や受理を表すものではない。
    import pandas as pd
    submission = pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    )
    submission.to_parquet("/kaggle/working/submission.parquet", index=False)
    print("Save & Run All placeholder created; no competition score was evaluated.")
